# Payments Data Analysis


Dataset created by Caixabank Tech for the 2024 AI Hackathon
Time Period: 2010s decade

1. Transaction Data (transactions_data.csv)
Detailed transaction records including amounts, timestamps, and merchant details
Covers transactions throughout the 2010s
Features transaction types, amounts, and merchant information
Perfect for analyzing spending patterns and building fraud detection models

2. Card Information (cards_dat.csv)
Credit and debit card details
Includes card limits, types, and activation dates
Links to customer accounts via card_id
Essential for understanding customer financial profiles

3. User Data (users_data)
Demographic information about customers
Account-related details
Enables customer segmentation and personalized analysis

Installing packages

In [0]:
%pip install kagglehub

In [0]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("computingvictor/transactions-fraud-datasets")

print("Path to dataset files:", path)

In [0]:
import pandas as pd
import json
transactions = pd.read_csv(f"{path}/transactions_data.csv")
cards = pd.read_csv(f"{path}/cards_data.csv")
users = pd.read_csv(f"{path}/users_data.csv")
with open(f"{path}/train_fraud_labels.json", "r") as f:
    raw = json.load(f)

# Flatten the nested structure: {"target": {"7476734": "No", ...}}
labels = pd.DataFrame(
    list(raw["target"].items()),
    columns=["transaction_id", "is_fraud"]
)

labels["transaction_id"] = labels["transaction_id"].astype(int)

In [0]:
transactions_2010 = transactions[transactions['date'].str.startswith('2010')].copy()
transactions_2010['amount'] = transactions_2010['amount'].str[1:].astype(float)
transactions_2010_spark = spark.createDataFrame(transactions_2010)
transactions_2010_spark.write.saveAsTable("pay_transactions_2010")

In [0]:
%sql
DROP TABLE pay_transactions_2010;

In [0]:
cards_spark = spark.createDataFrame(cards)
users_spark = spark.createDataFrame(users)
cards_spark.write.saveAsTable("pay_cards")
users_spark.write.saveAsTable("pay_users")
labels_spark = spark.createDataFrame(labels)
labels_spark.write.saveAsTable("pay_labels")

In [0]:
# %sql
# SELECT year(date),month(date),sum(cast(substring(amount, 2) AS decimal(10,2))) as total FROM pay_transactions_2010
# GROUP BY 1,2;

In [0]:
%sql
SELECT *
FROM workspace.default.pay_transactions_2010 trx
LEFT JOIN workspace.default.pay_users usr 
ON trx.client_id = usr.id
LEFT JOIN workspace.default.pay_cards crd
ON trx.card_id = crd.id

In [0]:
%sql
SELECT trx.date, month(trx.date), crd.client_id, amount, 
SUM(amount) OVER (PARTITION BY  crd.client_id)  TOT_SUM,
SUM(amount) OVER (PARTITION BY month(date), crd.client_id) MONTHLY_TOT,
amount/ monthly_tot *100 as perc_trx
FROM workspace.default.pay_transactions_2010 trx
LEFT JOIN workspace.default.pay_users usr 
ON trx.client_id = usr.id
LEFT JOIN workspace.default.pay_cards crd
ON trx.card_id = crd.id
WHERE usr.id = 825
ORDER BY month(trx.date) ASC, perc_trx DESC
;


In [0]:
%sql
-- ##### PROFILE CLIENT 825.... 
-- # Per ogni mese 2 maggiori transazioni. del cliente 825. Invetiate the card usage. And in which country he spends the most. 

## Fraud Detection ML Pipeline

This section implements a complete fraud detection system using **Isolation Forest** algorithm for anomaly detection.

### Pipeline Steps:

#### 1. Feature Engineering (SQL)
Creates `fraud_detection_features` table with:
* **Temporal Features**: year, month, day, hour, day_of_week, is_weekend, is_night
* **Card Features**: card_brand, card_type, credit_limit, has_chip, card_on_dark_web, amount_to_limit_ratio
* **Behavioral Features**: 
  * User-level: avg/stddev/min/max amounts, total transactions
  * Card-level: avg amount, transaction count
  * Monthly aggregations: total amount, transaction count
* **Anomaly Indicators**: z_score_amount, high_limit_usage, is_amount_outlier

#### 2. ML Preparation (Python)
* Loads feature table and converts to pandas
* Encodes categorical variables (card_brand, card_type)
* Handles missing values with median imputation
* Scales features using StandardScaler
* Total: **27 numeric features**

#### 3. Model Training (Isolation Forest)
* Algorithm: Isolation Forest (unsupervised anomaly detection)
* Contamination: 1% (assumes 1% fraud rate)
* 100 estimators for robust detection
* Tracked in MLflow for reproducibility

#### 4. Analysis & Visualization
* Fraud pattern analysis by card type, brand, time
* Comparative statistics (fraud vs normal)
* Visualizations: anomaly scores, amounts, temporal patterns

#### 5. Output
* `fraud_predictions` table with:
  * All original features
  * `is_fraud`: Binary flag (1=fraud, 0=normal)
  * `anomaly_score`: Continuous anomaly score

### Usage:
Run cells sequentially to generate fraud predictions. Review flagged transactions in the `fraud_predictions` table.

In [0]:
# %sql
# WITH trx_last_date (
#   SELECT *,
#   row_number() OVER (PARTITION BY client_id ORDER BY id DESC) rn
#   FROM pay_transactions_2010 AS trx
#   WHERE trx.client_id = 825
# )
# SELECT * FROM 
# trx_last_date trx
# INNER JOIN pay_users AS usr
# ON trx.client_id = usr.id
# WHERE rn = 1;  ---> Per evitare i doppioni




In [0]:
%sql
-- Comprehensive fraud detection feature set for Isolation Forest
CREATE OR REPLACE TABLE workspace.default.fraud_detection_features AS
WITH transaction_features AS (
  SELECT 
    trx.*,
    -- Temporal features
    year(trx.date) as year,
    month(trx.date) as month,
    dayofmonth(trx.date) as day,
    dayofweek(trx.date) as day_of_week,
    hour(trx.date) as hour,
    CASE WHEN dayofweek(trx.date) IN (1,7) THEN 1 ELSE 0 END as is_weekend,
    CASE WHEN hour(trx.date) BETWEEN 22 AND 6 THEN 1 ELSE 0 END as is_night,
    
    -- Card features
    crd.card_brand,
    crd.card_type,
    CAST(REPLACE(crd.credit_limit, '$', '') AS DOUBLE) as credit_limit,
    crd.has_chip,
    crd.num_cards_issued,
    CASE WHEN crd.card_on_dark_web = 'Yes' THEN 1 ELSE 0 END as card_on_dark_web,
    
    -- Amount to limit ratio
    trx.amount / NULLIF(CAST(REPLACE(crd.credit_limit, '$', '') AS DOUBLE), 0) as amount_to_limit_ratio,
    
    -- User aggregated features (behavioral patterns)
    AVG(trx.amount) OVER (PARTITION BY trx.client_id) as user_avg_amount,
    STDDEV(trx.amount) OVER (PARTITION BY trx.client_id) as user_stddev_amount,
    MIN(trx.amount) OVER (PARTITION BY trx.client_id) as user_min_amount,
    MAX(trx.amount) OVER (PARTITION BY trx.client_id) as user_max_amount,
    COUNT(*) OVER (PARTITION BY trx.client_id) as user_total_transactions,
    
    -- Card-level aggregations
    AVG(trx.amount) OVER (PARTITION BY trx.card_id) as card_avg_amount,
    COUNT(*) OVER (PARTITION BY trx.card_id) as card_total_transactions,
    
    -- Time-based aggregations (monthly)
    SUM(trx.amount) OVER (PARTITION BY trx.client_id, year(trx.date), month(trx.date)) as monthly_total_amount,
    COUNT(*) OVER (PARTITION BY trx.client_id, year(trx.date), month(trx.date)) as monthly_transaction_count,
    
    -- Deviation from user's normal behavior
    (trx.amount - AVG(trx.amount) OVER (PARTITION BY trx.client_id)) / 
      NULLIF(STDDEV(trx.amount) OVER (PARTITION BY trx.client_id), 0) as z_score_amount
    
  FROM workspace.default.pay_transactions_2010 trx
  LEFT JOIN workspace.default.pay_cards crd ON trx.card_id = crd.id
  LEFT JOIN workspace.default.pay_users usr ON trx.client_id = usr.id
)
SELECT 
  *,
  -- Derived anomaly indicators
  CASE WHEN amount_to_limit_ratio > 0.8 THEN 1 ELSE 0 END as high_limit_usage,
  CASE WHEN ABS(z_score_amount) > 3 THEN 1 ELSE 0 END as is_amount_outlier,
  amount / NULLIF(user_avg_amount, 0) as amount_vs_user_avg_ratio
FROM transaction_features;

In [0]:
# Load the fraud detection dataset and prepare for Isolation Forest
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load the feature table
df = spark.table("workspace.default.fraud_detection_features").toPandas()

print(f"Dataset shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

# Select numeric features for Isolation Forest
numeric_features = [
    'amount', 'year', 'month', 'day', 'day_of_week', 'hour',
    'is_weekend', 'is_night', 'credit_limit', 'num_cards_issued',
    'card_on_dark_web', 'amount_to_limit_ratio',
    'user_avg_amount', 'user_stddev_amount', 'user_min_amount', 'user_max_amount',
    'user_total_transactions', 'card_avg_amount', 'card_total_transactions',
    'monthly_total_amount', 'monthly_transaction_count',
    'z_score_amount', 'high_limit_usage', 'is_amount_outlier',
    'amount_vs_user_avg_ratio'
]

# Encode categorical features
le_brand = LabelEncoder()
le_type = LabelEncoder()

df['card_brand_encoded'] = le_brand.fit_transform(df['card_brand'].fillna('Unknown'))
df['card_type_encoded'] = le_type.fit_transform(df['card_type'].fillna('Unknown'))

# Add encoded features to numeric list
numeric_features.extend(['card_brand_encoded', 'card_type_encoded'])

# Create ML-ready dataframe
df_ml = df[numeric_features + ['id', 'client_id', 'card_id', 'date']].copy()

# Handle missing values
df_ml = df_ml.fillna(df_ml.median(numeric_only=True))

print(f"\nML-ready dataset shape: {df_ml.shape}")
print(f"Features for Isolation Forest: {len(numeric_features)}")
print(f"\nFeature list:\n{numeric_features}")

# Save feature names for later use
feature_columns = numeric_features

In [0]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import mlflow
import mlflow.sklearn

# Scale features for better performance
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_ml[feature_columns])

# Set up MLflow experiment
mlflow.set_experiment("/Users/ibritics@gmail.com/payment-analysis/fraud-detection")

with mlflow.start_run(run_name="isolation_forest_fraud_detection") as run:
    # Train Isolation Forest
    # contamination: expected proportion of outliers (0.01 = 1% fraud rate is reasonable)
    iso_forest = IsolationForest(
        contamination=0.01,
        random_state=42,
        n_estimators=100,
        max_samples='auto',
        n_jobs=-1
    )
    
    # Fit and predict
    predictions = iso_forest.fit_predict(X_scaled)
    anomaly_scores = iso_forest.score_samples(X_scaled)
    
    # Add predictions to dataframe (-1 = anomaly/fraud, 1 = normal)
    df_ml['is_fraud'] = (predictions == -1).astype(int)
    df_ml['anomaly_score'] = anomaly_scores
    
    # Calculate metrics
    fraud_count = df_ml['is_fraud'].sum()
    fraud_rate = fraud_count / len(df_ml) * 100
    
    # Log parameters and metrics
    mlflow.log_param("contamination", 0.01)
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("num_features", len(feature_columns))
    mlflow.log_metric("fraud_count", fraud_count)
    mlflow.log_metric("fraud_rate_percent", fraud_rate)
    mlflow.log_metric("total_transactions", len(df_ml))
    
    # Log model
    mlflow.sklearn.log_model(iso_forest, "isolation_forest_model")
    mlflow.sklearn.log_model(scaler, "scaler")
    
    print(f"\n===== FRAUD DETECTION RESULTS =====")
    print(f"Total transactions: {len(df_ml):,}")
    print(f"Detected fraud cases: {fraud_count:,}")
    print(f"Fraud rate: {fraud_rate:.2f}%")
    print(f"\nModel trained and logged to MLflow")
    print(f"Run ID: {run.info.run_id}")

# Display top fraud cases by anomaly score
fraud_cases = df_ml[df_ml['is_fraud'] == 1].sort_values('anomaly_score')
print(f"\nTop 10 suspicious transactions (most anomalous):")
display(fraud_cases[['id', 'client_id', 'card_id', 'date', 'amount', 'anomaly_score']].head(10))

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Analyze fraud patterns
print("\n===== FRAUD PATTERNS ANALYSIS =====")

# Fraud by card type
print("\n1. Fraud distribution by card type:")
fraud_by_type = df[df_ml['is_fraud'] == 1]['card_type'].value_counts()
print(fraud_by_type)

# Fraud by card brand
print("\n2. Fraud distribution by card brand:")
fraud_by_brand = df[df_ml['is_fraud'] == 1]['card_brand'].value_counts()
print(fraud_by_brand)

# Fraud by time patterns
print("\n3. Fraud by hour of day:")
fraud_by_hour = df_ml[df_ml['is_fraud'] == 1]['hour'].value_counts().sort_index()
print(fraud_by_hour.head(10))

# High-risk features for fraud
print("\n4. Average feature values - Fraud vs Normal:")
fraud_df = df_ml[df_ml['is_fraud'] == 1]
normal_df = df_ml[df_ml['is_fraud'] == 0]

comparison_features = ['amount', 'amount_to_limit_ratio', 'z_score_amount', 
                       'user_total_transactions', 'is_night', 'is_weekend']

for feat in comparison_features:
    fraud_mean = fraud_df[feat].mean()
    normal_mean = normal_df[feat].mean()
    print(f"{feat:30s} | Fraud: {fraud_mean:10.2f} | Normal: {normal_mean:10.2f} | Ratio: {fraud_mean/normal_mean if normal_mean != 0 else 0:.2f}x")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Anomaly score distribution
axes[0, 0].hist(df_ml[df_ml['is_fraud'] == 0]['anomaly_score'], bins=50, alpha=0.7, label='Normal', color='green')
axes[0, 0].hist(df_ml[df_ml['is_fraud'] == 1]['anomaly_score'], bins=50, alpha=0.7, label='Fraud', color='red')
axes[0, 0].set_xlabel('Anomaly Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Anomaly Score Distribution')
axes[0, 0].legend()

# 2. Amount distribution
axes[0, 1].hist(df_ml[df_ml['is_fraud'] == 0]['amount'], bins=50, alpha=0.7, label='Normal', color='green', range=(0, df_ml['amount'].quantile(0.99)))
axes[0, 1].hist(df_ml[df_ml['is_fraud'] == 1]['amount'], bins=50, alpha=0.7, label='Fraud', color='red', range=(0, df_ml['amount'].quantile(0.99)))
axes[0, 1].set_xlabel('Transaction Amount')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Transaction Amount Distribution')
axes[0, 1].legend()

# 3. Fraud by hour
fraud_hourly = df_ml[df_ml['is_fraud'] == 1].groupby('hour').size()
normal_hourly = df_ml[df_ml['is_fraud'] == 0].groupby('hour').size()
axes[1, 0].bar(fraud_hourly.index, fraud_hourly.values, alpha=0.7, label='Fraud', color='red')
axes[1, 0].set_xlabel('Hour of Day')
axes[1, 0].set_ylabel('Transaction Count')
axes[1, 0].set_title('Fraud Transactions by Hour')
axes[1, 0].legend()

# 4. Card dark web correlation
dark_web_fraud = df[df_ml['is_fraud'] == 1]['card_on_dark_web'].value_counts()
axes[1, 1].bar(dark_web_fraud.index, dark_web_fraud.values, color=['green', 'red'])
axes[1, 1].set_xlabel('Card on Dark Web')
axes[1, 1].set_ylabel('Fraud Count')
axes[1, 1].set_title('Fraud vs Card Dark Web Exposure')

plt.tight_layout()
plt.show()

print("\n===== ANALYSIS COMPLETE =====")

In [0]:
# Save fraud predictions to a new table for further analysis
fraud_predictions_spark = spark.createDataFrame(df_ml)
fraud_predictions_spark.write.mode('overwrite').saveAsTable("workspace.default.fraud_predictions")

print("\n===== FRAUD PREDICTIONS SAVED =====")
print(f"Table: workspace.default.fraud_predictions")
print(f"Total records: {len(df_ml):,}")
print(f"\nColumns:")
print(f"  - All original features ({len(feature_columns)} features)")
print(f"  - is_fraud: Binary flag (1=fraud, 0=normal)")
print(f"  - anomaly_score: Lower scores = more anomalous")
print(f"  - id, client_id, card_id, date: Transaction identifiers")

# Create summary statistics
print("\n===== SUMMARY STATISTICS =====")
print(f"\nFraud Detection Model Summary:")
print(f"  Algorithm: Isolation Forest")
print(f"  Features: {len(feature_columns)}")
print(f"  Contamination: 1%")
print(f"  Estimators: 100")
print(f"\nDataset Summary:")
print(f"  Total transactions: {len(df_ml):,}")
print(f"  Flagged as fraud: {df_ml['is_fraud'].sum():,} ({df_ml['is_fraud'].mean()*100:.2f}%)")
print(f"  Normal transactions: {(df_ml['is_fraud']==0).sum():,}")
print(f"\nKey Features Engineered:")
print(f"  - Temporal: hour, day_of_week, is_weekend, is_night")
print(f"  - Behavioral: user_avg_amount, user_stddev_amount, z_score_amount")
print(f"  - Card-specific: card_brand, card_type, credit_limit, card_on_dark_web")
print(f"  - Transaction: amount, amount_to_limit_ratio, monthly patterns")
print(f"  - Aggregations: user/card transaction counts and totals")
print(f"\nNext Steps:")
print(f"  1. Review high-risk transactions in fraud_predictions table")
print(f"  2. Investigate patterns in flagged transactions")
print(f"  3. Consider additional features or model tuning")
print(f"  4. Set up monitoring and alerting for new transactions")

In [0]:
%sql
SELECT count(*) FROM workspace.default.fraud_predictions A
LEFT JOIN workspace.default.pay_labels B
WHERE B.transaction_id is null;

In [0]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Join predictions with actual labels - ONLY for transactions in pay_transactions_2010
query = """
SELECT 
    pred.id as transaction_id,
    pred.is_fraud as predicted_fraud,
    CASE WHEN labels.is_fraud = 'Yes' THEN 1 ELSE 0 END as actual_fraud,
    pred.anomaly_score,
    pred.amount,
    pred.client_id,
    pred.card_id
FROM workspace.default.fraud_predictions pred
INNER JOIN workspace.default.pay_labels labels
    ON pred.id = labels.transaction_id
INNER JOIN workspace.default.pay_transactions_2010 trx
    ON pred.id = trx.id
"""

eval_df = spark.sql(query).toPandas()

print(f"Total transactions with labels (2010 only): {len(eval_df):,}")
print(f"\nActual fraud cases: {eval_df['actual_fraud'].sum():,}")
print(f"Predicted fraud cases: {eval_df['predicted_fraud'].sum():,}")

# Create confusion matrix
y_true = eval_df['actual_fraud']
y_pred = eval_df['predicted_fraud']

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()

print("\n" + "="*50)
print("CONFUSION MATRIX")
print("="*50)
print(f"\nTrue Negatives (TN):  {tn:,} - Correctly identified as normal")
print(f"False Positives (FP): {fp:,} - Normal flagged as fraud")
print(f"False Negatives (FN): {fn:,} - Fraud missed by model")
print(f"True Positives (TP):  {tp:,} - Correctly identified as fraud")

# Calculate metrics
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (tp + tn) / (tp + tn + fp + fn)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

print("\n" + "="*50)
print("PERFORMANCE METRICS")
print("="*50)
print(f"Accuracy:    {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision:   {precision:.4f} ({precision*100:.2f}%) - Of flagged transactions, how many are actually fraud")
print(f"Recall:      {recall:.4f} ({recall*100:.2f}%) - Of all fraud cases, how many we detected")
print(f"F1-Score:    {f1_score:.4f}")
print(f"Specificity: {specificity:.4f} ({specificity*100:.2f}%) - Of normal transactions, how many we correctly identified")

# Detailed classification report
print("\n" + "="*50)
print("DETAILED CLASSIFICATION REPORT")
print("="*50)
print(classification_report(y_true, y_pred, target_names=['Normal', 'Fraud']))

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
            xticklabels=['Normal', 'Fraud'], 
            yticklabels=['Normal', 'Fraud'])
axes[0, 0].set_ylabel('Actual')
axes[0, 0].set_xlabel('Predicted')
axes[0, 0].set_title('Confusion Matrix')

# 2. Normalized Confusion Matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', ax=axes[0, 1],
            xticklabels=['Normal', 'Fraud'], 
            yticklabels=['Normal', 'Fraud'])
axes[0, 1].set_ylabel('Actual')
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_title('Normalized Confusion Matrix (%)')

# 3. Metrics Bar Chart
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Specificity']
metrics_values = [accuracy, precision, recall, f1_score, specificity]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
axes[1, 0].bar(metrics_names, metrics_values, color=colors)
axes[1, 0].set_ylim([0, 1])
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_title('Model Performance Metrics')
for i, v in enumerate(metrics_values):
    axes[1, 0].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')

# 4. ROC Curve (using anomaly scores)
# Convert anomaly scores to probabilities (lower score = higher fraud probability)
fraud_prob = 1 / (1 + np.exp(eval_df['anomaly_score']))  # Sigmoid transformation
fpr, tpr, thresholds = roc_curve(y_true, fraud_prob)
roc_auc = roc_auc_score(y_true, fraud_prob)

axes[1, 1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1, 1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random classifier')
axes[1, 1].set_xlim([0.0, 1.0])
axes[1, 1].set_ylim([0.0, 1.05])
axes[1, 1].set_xlabel('False Positive Rate')
axes[1, 1].set_ylabel('True Positive Rate')
axes[1, 1].set_title('ROC Curve')
axes[1, 1].legend(loc='lower right')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nROC AUC Score: {roc_auc:.4f}")

# Save evaluation results
eval_df.to_csv('/tmp/fraud_evaluation.csv', index=False)
print("\nEvaluation results saved to /tmp/fraud_evaluation.csv")

## Supervised Learning for Fraud Detection

Now that we have ground truth labels from [pay_labels](#table), we can train supervised classification models and compare them to the unsupervised Isolation Forest approach.

### Models to Train:
1. **Logistic Regression** - Fast, interpretable baseline
2. **Random Forest** - Ensemble method, handles non-linear relationships
3. **XGBoost** - Gradient boosting, typically best performance

### Approach:
* Load features with actual fraud labels
* Split into train (70%) and test (30%) sets
* Train all three models
* Evaluate with confusion matrix, precision, recall, F1-score, ROC-AUC
* Compare supervised vs unsupervised approaches

In [0]:
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

# Load features with actual labels
query = """
SELECT 
    feat.*,
    CASE WHEN labels.is_fraud = 'Yes' THEN 1 ELSE 0 END as actual_fraud
FROM workspace.default.fraud_detection_features feat
INNER JOIN workspace.default.pay_labels labels
    ON feat.id = labels.transaction_id
INNER JOIN workspace.default.pay_transactions_2010 trx
    ON feat.id = trx.id
"""

df_supervised = spark.sql(query).toPandas()

print(f"Dataset with labels: {df_supervised.shape}")
print(f"\nFraud distribution:")
print(df_supervised['actual_fraud'].value_counts())
print(f"\nFraud rate: {df_supervised['actual_fraud'].mean()*100:.2f}%")

# Use the same feature engineering as Isolation Forest
from sklearn.preprocessing import LabelEncoder

# Encode categorical features
le_brand = LabelEncoder()
le_type = LabelEncoder()

df_supervised['card_brand_encoded'] = le_brand.fit_transform(df_supervised['card_brand'].fillna('Unknown'))
df_supervised['card_type_encoded'] = le_type.fit_transform(df_supervised['card_type'].fillna('Unknown'))

# Select features
feature_cols = [
    'amount', 'year', 'month', 'day', 'day_of_week', 'hour',
    'is_weekend', 'is_night', 'credit_limit', 'num_cards_issued',
    'card_on_dark_web', 'amount_to_limit_ratio',
    'user_avg_amount', 'user_stddev_amount', 'user_min_amount', 'user_max_amount',
    'user_total_transactions', 'card_avg_amount', 'card_total_transactions',
    'monthly_total_amount', 'monthly_transaction_count',
    'z_score_amount', 'high_limit_usage', 'is_amount_outlier',
    'amount_vs_user_avg_ratio', 'card_brand_encoded', 'card_type_encoded'
]

# Prepare X and y
X = df_supervised[feature_cols].copy()
y = df_supervised['actual_fraud'].copy()

# Handle missing values
X = X.fillna(X.median())

# Split into train and test sets (70/30 split, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\n===== TRAIN/TEST SPLIT =====")
print(f"Training set: {len(X_train):,} samples")
print(f"  - Fraud: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"  - Normal: {(y_train==0).sum():,} ({(y_train==0).mean()*100:.2f}%)")
print(f"\nTest set: {len(X_test):,} samples")
print(f"  - Fraud: {y_test.sum():,} ({y_test.mean()*100:.2f}%)")
print(f"  - Normal: {(y_test==0).sum():,} ({(y_test==0).mean()*100:.2f}%)")
print(f"\nFeatures: {len(feature_cols)}")

# Store transaction IDs for later analysis
train_ids = df_supervised.loc[X_train.index, 'id']
test_ids = df_supervised.loc[X_test.index, 'id']

In [0]:
%pip install xgboost scikit-learn imbalanced-learn

## Advanced Model Optimization

This section enhances the baseline supervised models with three critical techniques:

### 1. **SMOTE (Synthetic Minority Over-sampling Technique)**
* **Problem**: Fraud is rare (~1-5% of transactions) causing class imbalance
* **Solution**: Generate synthetic fraud samples to balance the training data
* **Benefit**: Prevents models from being biased toward the majority class

### 2. **Grid Search**
* **Problem**: Default hyperparameters may not be optimal for this dataset
* **Solution**: Systematically test multiple hyperparameter combinations
* **Tested Parameters**:
  * **Logistic Regression**: C (regularization), penalty (l1/l2/elasticnet)
  * **Random Forest**: n_estimators, max_depth, min_samples_split
  * **XGBoost**: n_estimators, max_depth, learning_rate, subsample

### 3. **Cross-Validation (5-Fold Stratified)**
* **Problem**: Single train/test split may not represent model's true performance
* **Solution**: Evaluate each model on 5 different train/validation splits
* **Benefit**: More robust performance estimates, reduces overfitting risk

### Expected Improvements:
* **Recall**: Better fraud detection (fewer false negatives)
* **Precision**: Fewer false alarms (fewer false positives)
* **F1-Score**: Balanced improvement in overall performance
* **Generalization**: Models perform consistently across different data splits

In [0]:
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np

print("="*70)
print("APPLYING SMOTE (Synthetic Minority Over-sampling Technique)")
print("="*70)

print(f"\nOriginal Training Set:")
print(f"  Total samples: {len(X_train):,}")
print(f"  Fraud cases: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"  Normal cases: {(y_train==0).sum():,} ({(y_train==0).mean()*100:.2f}%)")
print(f"  Imbalance ratio: {(y_train==0).sum() / y_train.sum():.1f}:1")

# Apply SMOTE
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE:")
print(f"  Total samples: {len(X_train_smote):,}")
print(f"  Fraud cases: {y_train_smote.sum():,} ({y_train_smote.mean()*100:.2f}%)")
print(f"  Normal cases: {(y_train_smote==0).sum():,} ({(y_train_smote==0).mean()*100:.2f}%)")
print(f"  Imbalance ratio: {(y_train_smote==0).sum() / y_train_smote.sum():.1f}:1")

print(f"\n✓ SMOTE applied successfully - training set is now balanced")
print(f"  Synthetic samples created: {len(X_train_smote) - len(X_train):,}")

# Scale features for Logistic Regression
from sklearn.preprocessing import StandardScaler
scaler_smote = StandardScaler()
X_train_smote_scaled = scaler_smote.fit_transform(X_train_smote)
X_test_scaled = scaler_smote.transform(X_test)

In [0]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import time

print("="*70)
print("HYPERPARAMETER TUNING WITH GRID SEARCH + CROSS-VALIDATION")
print("="*70)

# Set up cross-validation strategy
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Set MLflow experiment
mlflow.set_experiment("/Users/ibritics@gmail.com/payment-analysis/fraud-detection-gridsearch")

# Store best models and results
best_models = {}
grid_search_results = []

# 1. LOGISTIC REGRESSION GRID SEARCH
print("\n[1/3] Grid Search - Logistic Regression")
print("   Testing 12 parameter combinations with 5-fold CV (60 fits total)...")
start_time = time.time()

with mlflow.start_run(run_name="lr_gridsearch") as run:
    param_grid_lr = {
        'C': [0.01, 0.1, 1.0, 10.0],
        'penalty': ['l1', 'l2', 'elasticnet'],
        'solver': ['saga']
    }
    
    lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
    
    grid_lr = GridSearchCV(
        lr, param_grid_lr, cv=cv_strategy,
        scoring='f1', n_jobs=-1, verbose=0
    )
    
    grid_lr.fit(X_train_smote_scaled, y_train_smote)
    
    # Get best model and predictions
    best_lr = grid_lr.best_estimator_
    y_pred_lr = best_lr.predict(X_test_scaled)
    y_proba_lr = best_lr.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    from sklearn.metrics import confusion_matrix, roc_auc_score
    cm = confusion_matrix(y_test, y_pred_lr)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    roc_auc = roc_auc_score(y_test, y_proba_lr)
    
    # Log to MLflow
    mlflow.log_params(grid_lr.best_params_)
    mlflow.log_metrics({
        'best_cv_f1': grid_lr.best_score_,
        'test_f1': f1,
        'test_roc_auc': roc_auc,
        'test_precision': precision,
        'test_recall': recall
    })
    mlflow.sklearn.log_model(best_lr, "model")
    
    best_models['Logistic Regression (GridSearch)'] = {
        'model': best_lr,
        'predictions': y_pred_lr,
        'probabilities': y_proba_lr,
        'best_params': grid_lr.best_params_,
        'cv_f1': grid_lr.best_score_
    }
    
    elapsed = time.time() - start_time
    print(f"   ✓ Completed in {elapsed:.1f}s")
    print(f"   Best params: {grid_lr.best_params_}")
    print(f"   CV F1: {grid_lr.best_score_:.4f} | Test F1: {f1:.4f} | Test ROC-AUC: {roc_auc:.4f}")

# 2. RANDOM FOREST GRID SEARCH
print("\n[2/3] Grid Search - Random Forest")
print("   Testing 36 parameter combinations with 5-fold CV (180 fits total)...")
start_time = time.time()

with mlflow.start_run(run_name="rf_gridsearch") as run:
    param_grid_rf = {
        'n_estimators': [50, 100, 200],
        'max_depth': [5, 10, 15, None],
        'min_samples_split': [2, 5, 10]
    }
    
    rf = RandomForestClassifier(random_state=42, n_jobs=-1)
    
    grid_rf = GridSearchCV(
        rf, param_grid_rf, cv=cv_strategy,
        scoring='f1', n_jobs=-1, verbose=0
    )
    
    grid_rf.fit(X_train_smote, y_train_smote)
    
    # Get best model and predictions
    best_rf = grid_rf.best_estimator_
    y_pred_rf = best_rf.predict(X_test)
    y_proba_rf = best_rf.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    cm = confusion_matrix(y_test, y_pred_rf)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    roc_auc = roc_auc_score(y_test, y_proba_rf)
    
    # Log to MLflow
    mlflow.log_params(grid_rf.best_params_)
    mlflow.log_metrics({
        'best_cv_f1': grid_rf.best_score_,
        'test_f1': f1,
        'test_roc_auc': roc_auc,
        'test_precision': precision,
        'test_recall': recall
    })
    mlflow.sklearn.log_model(best_rf, "model")
    
    best_models['Random Forest (GridSearch)'] = {
        'model': best_rf,
        'predictions': y_pred_rf,
        'probabilities': y_proba_rf,
        'best_params': grid_rf.best_params_,
        'cv_f1': grid_rf.best_score_
    }
    
    elapsed = time.time() - start_time
    print(f"   ✓ Completed in {elapsed:.1f}s")
    print(f"   Best params: {grid_rf.best_params_}")
    print(f"   CV F1: {grid_rf.best_score_:.4f} | Test F1: {f1:.4f} | Test ROC-AUC: {roc_auc:.4f}")

# 3. XGBOOST GRID SEARCH
print("\n[3/3] Grid Search - XGBoost")
print("   Testing 48 parameter combinations with 5-fold CV (240 fits total)...")
start_time = time.time()

with mlflow.start_run(run_name="xgb_gridsearch") as run:
    param_grid_xgb = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.8, 1.0]
    }
    
    xgb = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')
    
    grid_xgb = GridSearchCV(
        xgb, param_grid_xgb, cv=cv_strategy,
        scoring='f1', n_jobs=-1, verbose=0
    )
    
    grid_xgb.fit(X_train_smote, y_train_smote)
    
    # Get best model and predictions
    best_xgb = grid_xgb.best_estimator_
    y_pred_xgb = best_xgb.predict(X_test)
    y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    cm = confusion_matrix(y_test, y_pred_xgb)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    roc_auc = roc_auc_score(y_test, y_proba_xgb)
    
    # Log to MLflow
    mlflow.log_params(grid_xgb.best_params_)
    mlflow.log_metrics({
        'best_cv_f1': grid_xgb.best_score_,
        'test_f1': f1,
        'test_roc_auc': roc_auc,
        'test_precision': precision,
        'test_recall': recall
    })
    mlflow.xgboost.log_model(best_xgb, "model")
    
    best_models['XGBoost (GridSearch)'] = {
        'model': best_xgb,
        'predictions': y_pred_xgb,
        'probabilities': y_proba_xgb,
        'best_params': grid_xgb.best_params_,
        'cv_f1': grid_xgb.best_score_
    }
    
    elapsed = time.time() - start_time
    print(f"   ✓ Completed in {elapsed:.1f}s")
    print(f"   Best params: {grid_xgb.best_params_}")
    print(f"   CV F1: {grid_xgb.best_score_:.4f} | Test F1: {f1:.4f} | Test ROC-AUC: {roc_auc:.4f}")

print("\n" + "="*70)
print("GRID SEARCH COMPLETE - ALL MODELS OPTIMIZED")
print("="*70)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, classification_report
import pandas as pd
import numpy as np

print("="*70)
print("OPTIMIZED MODELS COMPARISON (SMOTE + GridSearch + 5-Fold CV)")
print("="*70)

# Create comprehensive comparison
optimized_comparison = []

for model_name, model_info in best_models.items():
    y_pred = model_info['predictions']
    y_proba = model_info['probabilities']
    
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    roc_auc = roc_auc_score(y_test, y_proba)
    
    optimized_comparison.append({
        'Model': model_name,
        'TP': tp,
        'FP': fp,
        'FN': fn,
        'TN': tn,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Specificity': specificity,
        'ROC-AUC': roc_auc,
        'CV F1-Score': model_info['cv_f1']
    })
    
    print(f"\n{model_name}:")
    print(f"  Best Params: {model_info['best_params']}")
    print(f"  CV F1: {model_info['cv_f1']:.4f} | Test F1: {f1:.4f}")
    print(f"  TP: {tp:,}  |  FP: {fp:,}  |  FN: {fn:,}  |  TN: {tn:,}")
    print(f"  Precision: {precision:.4f}  |  Recall: {recall:.4f}  |  ROC-AUC: {roc_auc:.4f}")

optimized_df = pd.DataFrame(optimized_comparison)

print("\n" + "="*70)
print("SUMMARY TABLE - OPTIMIZED MODELS")
print("="*70)
display(optimized_df)

# Visualizations
fig = plt.figure(figsize=(18, 12))

# 1-3. Confusion Matrices
for idx, model_name in enumerate(best_models.keys()):
    ax = plt.subplot(3, 3, idx + 1)
    cm = confusion_matrix(y_test, best_models[model_name]['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Fraud'], 
                yticklabels=['Normal', 'Fraud'])
    ax.set_ylabel('Actual')
    ax.set_xlabel('Predicted')
    short_name = model_name.replace(' (GridSearch)', '')
    ax.set_title(f'{short_name}\nConfusion Matrix')

# 4. Metrics Comparison
ax4 = plt.subplot(3, 3, 4)
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x_pos = np.arange(len(metrics_to_plot))
width = 0.25
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for idx, model_name in enumerate(best_models.keys()):
    values = [optimized_df[optimized_df['Model'] == model_name][metric].values[0] 
              for metric in metrics_to_plot]
    ax4.bar(x_pos + idx * width, values, width, label=model_name.replace(' (GridSearch)', ''), 
            color=colors[idx], alpha=0.8)

ax4.set_ylabel('Score')
ax4.set_title('Performance Metrics Comparison')
ax4.set_xticks(x_pos + width)
ax4.set_xticklabels(metrics_to_plot, rotation=45, ha='right')
ax4.legend()
ax4.set_ylim([0, 1])
ax4.grid(axis='y', alpha=0.3)

# 5. ROC Curves
ax5 = plt.subplot(3, 3, 5)
colors_roc = ['#1f77b4', '#ff7f0e', '#2ca02c']

for idx, (model_name, model_info) in enumerate(best_models.items()):
    y_proba = model_info['probabilities']
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)
    short_name = model_name.replace(' (GridSearch)', '')
    ax5.plot(fpr, tpr, color=colors_roc[idx], lw=2, 
             label=f'{short_name} (AUC = {roc_auc:.3f})')

ax5.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
ax5.set_xlim([0.0, 1.0])
ax5.set_ylim([0.0, 1.05])
ax5.set_xlabel('False Positive Rate')
ax5.set_ylabel('True Positive Rate')
ax5.set_title('ROC Curves - Optimized Models')
ax5.legend(loc='lower right')
ax5.grid(alpha=0.3)

# 6. CV vs Test F1 Comparison
ax6 = plt.subplot(3, 3, 6)
models_list = [m.replace(' (GridSearch)', '') for m in optimized_df['Model']]
x_pos = np.arange(len(models_list))
width = 0.35

cv_f1 = optimized_df['CV F1-Score'].tolist()
test_f1 = optimized_df['F1-Score'].tolist()

ax6.bar(x_pos - width/2, cv_f1, width, label='CV F1-Score', color='skyblue', alpha=0.8)
ax6.bar(x_pos + width/2, test_f1, width, label='Test F1-Score', color='coral', alpha=0.8)

ax6.set_ylabel('F1-Score')
ax6.set_title('Cross-Validation vs Test Set Performance')
ax6.set_xticks(x_pos)
ax6.set_xticklabels(models_list, rotation=45, ha='right')
ax6.legend()
ax6.set_ylim([0, 1])
ax6.grid(axis='y', alpha=0.3)

# 7. Recall Comparison
ax7 = plt.subplot(3, 3, 7)
recall_data = optimized_df[['Model', 'Recall']].copy()
recall_data['Model'] = recall_data['Model'].str.replace(' (GridSearch)', '')
recall_data = recall_data.sort_values('Recall', ascending=False)
ax7.barh(recall_data['Model'], recall_data['Recall'], color=['#2ca02c', '#ff7f0e', '#1f77b4'])
ax7.set_xlabel('Recall (True Positive Rate)')
ax7.set_title('Recall Comparison\n(Higher = Fewer Missed Frauds)')
ax7.set_xlim([0, 1])
for i, v in enumerate(recall_data['Recall']):
    ax7.text(v + 0.02, i, f'{v:.3f}', va='center')
ax7.grid(axis='x', alpha=0.3)

# 8. Precision Comparison
ax8 = plt.subplot(3, 3, 8)
precision_data = optimized_df[['Model', 'Precision']].copy()
precision_data['Model'] = precision_data['Model'].str.replace(' (GridSearch)', '')
precision_data = precision_data.sort_values('Precision', ascending=False)
ax8.barh(precision_data['Model'], precision_data['Precision'], color=['#ff7f0e', '#2ca02c', '#1f77b4'])
ax8.set_xlabel('Precision')
ax8.set_title('Precision Comparison\n(Higher = Fewer False Alarms)')
ax8.set_xlim([0, 1])
for i, v in enumerate(precision_data['Precision']):
    ax8.text(v + 0.02, i, f'{v:.3f}', va='center')
ax8.grid(axis='x', alpha=0.3)

# 9. TP/FP/FN Comparison
ax9 = plt.subplot(3, 3, 9)
models_list_full = [m.replace(' (GridSearch)', '') for m in optimized_df['Model']]
x_pos = np.arange(len(models_list_full))
width = 0.25

tp_values = optimized_df['TP'].tolist()
fp_values = optimized_df['FP'].tolist()
fn_values = optimized_df['FN'].tolist()

ax9.bar(x_pos - width, tp_values, width, label='True Positives', color='green', alpha=0.8)
ax9.bar(x_pos, fp_values, width, label='False Positives', color='orange', alpha=0.8)
ax9.bar(x_pos + width, fn_values, width, label='False Negatives', color='red', alpha=0.8)

ax9.set_ylabel('Count')
ax9.set_title('TP / FP / FN Comparison')
ax9.set_xticks(x_pos)
ax9.set_xticklabels([m.replace(' ', '\n') for m in models_list_full])
ax9.legend()
ax9.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Identify best model
best_model_name = optimized_df.loc[optimized_df['F1-Score'].idxmax(), 'Model']
best_f1 = optimized_df['F1-Score'].max()
best_roc_auc = optimized_df.loc[optimized_df['Model'] == best_model_name, 'ROC-AUC'].values[0]

print("\n" + "="*70)
print(f"BEST OPTIMIZED MODEL: {best_model_name}")
print(f"  Test F1-Score: {best_f1:.4f}")
print(f"  Test ROC-AUC: {best_roc_auc:.4f}")
print("="*70)

# Detailed classification report for best model
print(f"\nDetailed Classification Report - {best_model_name}:")
y_pred_best = best_models[best_model_name]['predictions']
print(classification_report(y_test, y_pred_best, target_names=['Normal', 'Fraud']))

print("\n" + "="*70)
print("KEY IMPROVEMENTS WITH SMOTE + GridSearch + CV")
print("="*70)
print("  ✓ SMOTE: Balanced training data (1:1 ratio)")
print("  ✓ GridSearch: Found optimal hyperparameters")
print("  ✓ 5-Fold CV: Robust performance estimation")
print("  ✓ All models logged to MLflow for tracking")

In [0]:
import pandas as pd

print("="*70)
print("SAVING BEST OPTIMIZED MODEL PREDICTIONS")
print("="*70)

# Get best optimized model
best_opt_model_name = optimized_df.loc[optimized_df['F1-Score'].idxmax(), 'Model']
best_opt_model_info = best_models[best_opt_model_name]
best_opt_f1 = optimized_df['F1-Score'].max()
best_opt_roc = optimized_df.loc[optimized_df['Model'] == best_opt_model_name, 'ROC-AUC'].values[0]

print(f"\nBest Optimized Model: {best_opt_model_name}")
print(f"  F1-Score: {best_opt_f1:.4f}")
print(f"  ROC-AUC: {best_opt_roc:.4f}")
print(f"  Best Hyperparameters: {best_opt_model_info['best_params']}")

# Create predictions dataframe
opt_predictions_df = pd.DataFrame({
    'transaction_id': test_ids.values,
    'predicted_fraud': best_opt_model_info['predictions'],
    'fraud_probability': best_opt_model_info['probabilities'],
    'actual_fraud': y_test.values
})

# Add transaction details
opt_predictions_df = opt_predictions_df.merge(
    df_supervised[['id', 'client_id', 'card_id', 'amount', 'date']],
    left_on='transaction_id',
    right_on='id',
    how='left'
)

# Save to Spark table
opt_predictions_spark = spark.createDataFrame(opt_predictions_df)
opt_predictions_spark.write.mode('overwrite').saveAsTable("workspace.default.optimized_fraud_predictions")

print(f"\n✓ Predictions saved to: workspace.default.optimized_fraud_predictions")
print(f"  Total predictions: {len(opt_predictions_df):,}")
print(f"  Predicted frauds: {best_opt_model_info['predictions'].sum():,}")
print(f"  Actual frauds: {y_test.sum():,}")

# Show top high-risk transactions
print("\n" + "="*70)
print("TOP 10 HIGH-RISK TRANSACTIONS (by fraud probability)")
print("="*70)
high_risk_opt = opt_predictions_df.nlargest(10, 'fraud_probability')[[
    'transaction_id', 'client_id', 'amount', 'fraud_probability', 
    'predicted_fraud', 'actual_fraud'
]]
display(high_risk_opt)

# FINAL COMPARISON: Baseline vs Optimized
print("\n" + "="*70)
print("FINAL COMPARISON: BASELINE vs OPTIMIZED MODELS")
print("="*70)

if 'comparison_df' in globals():
    # Compare best baseline vs best optimized
    best_baseline_f1 = comparison_df['F1-Score'].max()
    best_baseline_model = comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'Model']
    best_baseline_recall = comparison_df.loc[comparison_df['Model'] == best_baseline_model, 'Recall'].values[0]
    best_baseline_precision = comparison_df.loc[comparison_df['Model'] == best_baseline_model, 'Precision'].values[0]
    
    best_opt_recall = optimized_df.loc[optimized_df['Model'] == best_opt_model_name, 'Recall'].values[0]
    best_opt_precision = optimized_df.loc[optimized_df['Model'] == best_opt_model_name, 'Precision'].values[0]
    
    print(f"\nBASELINE (Default hyperparameters, class_weight='balanced'):")
    print(f"  Best Model: {best_baseline_model}")
    print(f"  F1-Score: {best_baseline_f1:.4f}")
    print(f"  Precision: {best_baseline_precision:.4f}")
    print(f"  Recall: {best_baseline_recall:.4f}")
    
    print(f"\nOPTIMIZED (SMOTE + GridSearch + 5-Fold CV):")
    print(f"  Best Model: {best_opt_model_name}")
    print(f"  F1-Score: {best_opt_f1:.4f} ({'+' if best_opt_f1 > best_baseline_f1 else ''}{(best_opt_f1 - best_baseline_f1):.4f})")
    print(f"  Precision: {best_opt_precision:.4f} ({'+' if best_opt_precision > best_baseline_precision else ''}{(best_opt_precision - best_baseline_precision):.4f})")
    print(f"  Recall: {best_opt_recall:.4f} ({'+' if best_opt_recall > best_baseline_recall else ''}{(best_opt_recall - best_baseline_recall):.4f})")
    
    # Improvement percentages
    f1_improvement = ((best_opt_f1 - best_baseline_f1) / best_baseline_f1) * 100
    recall_improvement = ((best_opt_recall - best_baseline_recall) / best_baseline_recall) * 100
    precision_improvement = ((best_opt_precision - best_baseline_precision) / best_baseline_precision) * 100
    
    print(f"\nIMPROVEMENT:")
    print(f"  F1-Score: {f1_improvement:+.1f}%")
    print(f"  Precision: {precision_improvement:+.1f}%")
    print(f"  Recall: {recall_improvement:+.1f}%")

print("\n" + "="*70)
print("COMPLETE SUPERVISED LEARNING PIPELINE FINISHED")
print("="*70)
print("\n✓ Baseline models trained (Logistic Regression, Random Forest, XGBoost)")
print("✓ SMOTE applied for balanced training data")
print("✓ Grid Search with 5-Fold Cross-Validation completed")
print("✓ Optimized models trained with best hyperparameters")
print("✓ All experiments logged to MLflow")
print("✓ Predictions saved to tables:")
print("    - supervised_fraud_predictions (baseline models)")
print("    - optimized_fraud_predictions (SMOTE + GridSearch models)")
print("\nNext Steps:")
print("  1. Deploy best optimized model for production")
print("  2. Set up real-time fraud scoring pipeline")
print("  3. Monitor model performance over time")
print("  4. Consider ensemble methods combining multiple models")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import roc_curve, roc_auc_score

print("="*70)
print("COMPREHENSIVE ROC AUC COMPARISON - ALL MODELS")
print("="*70)

# Collect all ROC AUC scores
all_roc_auc_data = []

# 1. ISOLATION FOREST (Unsupervised)
print("\n[1] Isolation Forest (Unsupervised):")
if 'eval_df' in globals():
    # Convert anomaly scores to probabilities
    iso_fraud_prob = 1 / (1 + np.exp(eval_df['anomaly_score']))
    iso_roc_auc = roc_auc_score(eval_df['actual_fraud'], iso_fraud_prob)
    print(f"    ROC AUC: {iso_roc_auc:.4f}")
    all_roc_auc_data.append({
        'Model': 'Isolation Forest',
        'Category': 'Unsupervised',
        'ROC-AUC': iso_roc_auc,
        'Type': 'Baseline'
    })
else:
    print("    Note: Run confusion matrix cell first to evaluate Isolation Forest")

# 2. BASELINE SUPERVISED MODELS
print("\n[2] Baseline Supervised Models (class_weight='balanced'):")
if 'comparison_df' in globals():
    for idx, row in comparison_df.iterrows():
        model_name = row['Model']
        roc_auc = row['ROC-AUC']
        print(f"    {model_name}: {roc_auc:.4f}")
        all_roc_auc_data.append({
            'Model': model_name,
            'Category': 'Supervised (Baseline)',
            'ROC-AUC': roc_auc,
            'Type': 'Baseline'
        })
else:
    print("    Note: Run baseline model training cells to see results")

# 3. OPTIMIZED SUPERVISED MODELS (SMOTE + GridSearch)
print("\n[3] Optimized Supervised Models (SMOTE + GridSearch + 5-Fold CV):")
if 'optimized_df' in globals():
    for idx, row in optimized_df.iterrows():
        model_name = row['Model'].replace(' (GridSearch)', '')
        roc_auc = row['ROC-AUC']
        cv_f1 = row['CV F1-Score']
        print(f"    {model_name}: {roc_auc:.4f} (CV F1: {cv_f1:.4f})")
        all_roc_auc_data.append({
            'Model': model_name,
            'Category': 'Supervised (Optimized)',
            'ROC-AUC': roc_auc,
            'Type': 'Optimized'
        })
else:
    print("    Note: Run optimized model training cells to see results")

# Create comparison dataframe
if len(all_roc_auc_data) > 0:
    roc_comparison_df = pd.DataFrame(all_roc_auc_data)
    
    print("\n" + "="*70)
    print("ROC AUC SUMMARY TABLE")
    print("="*70)
    display(roc_comparison_df.sort_values('ROC-AUC', ascending=False))
    
    # Find best model by ROC AUC
    best_roc_idx = roc_comparison_df['ROC-AUC'].idxmax()
    best_roc_model = roc_comparison_df.loc[best_roc_idx, 'Model']
    best_roc_score = roc_comparison_df.loc[best_roc_idx, 'ROC-AUC']
    best_roc_category = roc_comparison_df.loc[best_roc_idx, 'Category']
    
    print("\n" + "="*70)
    print(f"BEST MODEL BY ROC AUC: {best_roc_model}")
    print(f"  Category: {best_roc_category}")
    print(f"  ROC AUC: {best_roc_score:.4f}")
    print("="*70)
    
    # VISUALIZATIONS
    fig = plt.figure(figsize=(18, 12))
    
    # 1. ROC AUC Bar Chart by Model
    ax1 = plt.subplot(2, 3, 1)
    sorted_df = roc_comparison_df.sort_values('ROC-AUC', ascending=True)
    colors = ['#e74c3c' if cat == 'Unsupervised' else '#3498db' if cat == 'Supervised (Baseline)' else '#2ecc71' 
              for cat in sorted_df['Category']]
    bars = ax1.barh(range(len(sorted_df)), sorted_df['ROC-AUC'], color=colors, alpha=0.8)
    ax1.set_yticks(range(len(sorted_df)))
    ax1.set_yticklabels(sorted_df['Model'])
    ax1.set_xlabel('ROC AUC Score')
    ax1.set_title('ROC AUC Comparison - All Models')
    ax1.set_xlim([0.5, 1.0])
    ax1.axvline(x=0.5, color='red', linestyle='--', alpha=0.3, label='Random Classifier')
    for i, v in enumerate(sorted_df['ROC-AUC']):
        ax1.text(v + 0.01, i, f'{v:.4f}', va='center')
    ax1.grid(axis='x', alpha=0.3)
    ax1.legend()
    
    # 2. ROC AUC by Category (Grouped)
    ax2 = plt.subplot(2, 3, 2)
    category_means = roc_comparison_df.groupby('Category')['ROC-AUC'].agg(['mean', 'max', 'min'])
    x_pos = np.arange(len(category_means))
    ax2.bar(x_pos, category_means['mean'], alpha=0.6, label='Mean', color='steelblue')
    ax2.scatter(x_pos, category_means['max'], color='green', s=100, marker='^', label='Max', zorder=3)
    ax2.scatter(x_pos, category_means['min'], color='red', s=100, marker='v', label='Min', zorder=3)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(category_means.index, rotation=45, ha='right')
    ax2.set_ylabel('ROC AUC Score')
    ax2.set_title('ROC AUC by Model Category')
    ax2.set_ylim([0.5, 1.0])
    ax2.legend()
    ax2.grid(axis='y', alpha=0.3)
    for i, (mean_val, max_val) in enumerate(zip(category_means['mean'], category_means['max'])):
        ax2.text(i, mean_val + 0.02, f'{mean_val:.3f}', ha='center', va='bottom')
    
    # 3. Improvement Over Random Classifier
    ax3 = plt.subplot(2, 3, 3)
    sorted_df['Improvement'] = (sorted_df['ROC-AUC'] - 0.5) * 100
    bars = ax3.barh(range(len(sorted_df)), sorted_df['Improvement'], color=colors, alpha=0.8)
    ax3.set_yticks(range(len(sorted_df)))
    ax3.set_yticklabels(sorted_df['Model'])
    ax3.set_xlabel('Improvement Over Random (%)')
    ax3.set_title('ROC AUC Improvement Over Random Classifier')
    for i, v in enumerate(sorted_df['Improvement']):
        ax3.text(v + 1, i, f'{v:.1f}%', va='center')
    ax3.grid(axis='x', alpha=0.3)
    
    # 4. Combined ROC Curves (if data available)
    ax4 = plt.subplot(2, 3, 4)
    
    # Plot Isolation Forest ROC
    if 'eval_df' in globals():
        iso_fraud_prob = 1 / (1 + np.exp(eval_df['anomaly_score']))
        fpr_iso, tpr_iso, _ = roc_curve(eval_df['actual_fraud'], iso_fraud_prob)
        ax4.plot(fpr_iso, tpr_iso, label=f'Isolation Forest (AUC={iso_roc_auc:.3f})', 
                 color='#e74c3c', linewidth=2)
    
    # Plot Baseline Supervised ROC curves
    if 'proba_predictions' in globals() and 'y_test' in globals():
        baseline_colors = {'Logistic Regression': '#3498db', 'Random Forest': '#9b59b6', 'XGBoost': '#1abc9c'}
        for model_name, y_proba in proba_predictions.items():
            fpr, tpr, _ = roc_curve(y_test, y_proba)
            roc_auc = roc_auc_score(y_test, y_proba)
            ax4.plot(fpr, tpr, label=f'{model_name} (AUC={roc_auc:.3f})', 
                     color=baseline_colors.get(model_name, 'gray'), linewidth=2, linestyle='--')
    
    # Plot Optimized ROC curves
    if 'best_models' in globals() and 'y_test' in globals():
        opt_colors = {'Logistic Regression': '#2ecc71', 'Random Forest': '#f39c12', 'XGBoost': '#e67e22'}
        for model_name, model_info in best_models.items():
            y_proba = model_info['probabilities']
            fpr, tpr, _ = roc_curve(y_test, y_proba)
            roc_auc = roc_auc_score(y_test, y_proba)
            short_name = model_name.replace(' (GridSearch)', '')
            ax4.plot(fpr, tpr, label=f'{short_name} Opt (AUC={roc_auc:.3f})', 
                     color=opt_colors.get(short_name, 'gray'), linewidth=2.5)
    
    ax4.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.500)')
    ax4.set_xlim([0.0, 1.0])
    ax4.set_ylim([0.0, 1.05])
    ax4.set_xlabel('False Positive Rate')
    ax4.set_ylabel('True Positive Rate')
    ax4.set_title('Combined ROC Curves - All Models')
    ax4.legend(loc='lower right', fontsize=8)
    ax4.grid(alpha=0.3)
    
    # 5. Baseline vs Optimized Comparison
    ax5 = plt.subplot(2, 3, 5)
    if 'comparison_df' in globals() and 'optimized_df' in globals():
        comparison_data = []
        for model in ['Logistic Regression', 'Random Forest', 'XGBoost']:
            baseline_roc = comparison_df[comparison_df['Model'] == model]['ROC-AUC'].values[0] if model in comparison_df['Model'].values else None
            opt_model_name = f"{model} (GridSearch)"
            opt_roc = optimized_df[optimized_df['Model'] == opt_model_name]['ROC-AUC'].values[0] if opt_model_name in optimized_df['Model'].values else None
            
            if baseline_roc is not None and opt_roc is not None:
                comparison_data.append({
                    'Model': model,
                    'Baseline': baseline_roc,
                    'Optimized': opt_roc,
                    'Improvement': opt_roc - baseline_roc
                })
        
        if comparison_data:
            comp_df = pd.DataFrame(comparison_data)
            x_pos = np.arange(len(comp_df))
            width = 0.35
            
            ax5.bar(x_pos - width/2, comp_df['Baseline'], width, label='Baseline', color='skyblue', alpha=0.8)
            ax5.bar(x_pos + width/2, comp_df['Optimized'], width, label='Optimized (SMOTE+GridSearch)', color='lightgreen', alpha=0.8)
            
            ax5.set_ylabel('ROC AUC Score')
            ax5.set_title('Baseline vs Optimized Models')
            ax5.set_xticks(x_pos)
            ax5.set_xticklabels(comp_df['Model'], rotation=45, ha='right')
            ax5.legend()
            ax5.set_ylim([0.5, 1.0])
            ax5.grid(axis='y', alpha=0.3)
            
            # Add improvement annotations
            for i, row in comp_df.iterrows():
                improvement = row['Improvement'] * 100
                if improvement > 0:
                    ax5.annotate(f'+{improvement:.1f}%', 
                                xy=(i, max(row['Baseline'], row['Optimized'])), 
                                xytext=(0, 5), textcoords='offset points',
                                ha='center', fontsize=9, color='green', weight='bold')
    
    # 6. Statistical Summary
    ax6 = plt.subplot(2, 3, 6)
    ax6.axis('off')
    
    summary_text = f"""
    ROC AUC STATISTICAL SUMMARY
    {'='*40}
    
    Total Models Evaluated: {len(roc_comparison_df)}
    
    Best Overall:
      • {best_roc_model}
      • ROC AUC: {best_roc_score:.4f}
    
    Category Performance:
    """
    
    for category in roc_comparison_df['Category'].unique():
        cat_data = roc_comparison_df[roc_comparison_df['Category'] == category]['ROC-AUC']
        summary_text += f"\n    {category}:"
        summary_text += f"\n      • Mean: {cat_data.mean():.4f}"
        summary_text += f"\n      • Best: {cat_data.max():.4f}"
        summary_text += f"\n      • Worst: {cat_data.min():.4f}"
    
    # Calculate improvement from baseline to optimized
    if 'comparison_df' in globals() and 'optimized_df' in globals():
        avg_baseline = comparison_df['ROC-AUC'].mean()
        avg_optimized = optimized_df['ROC-AUC'].mean()
        improvement_pct = ((avg_optimized - avg_baseline) / avg_baseline) * 100
        summary_text += f"\n\n    Optimization Impact:"
        summary_text += f"\n      • Avg Baseline: {avg_baseline:.4f}"
        summary_text += f"\n      • Avg Optimized: {avg_optimized:.4f}"
        summary_text += f"\n      • Improvement: {improvement_pct:+.1f}%"
    
    ax6.text(0.1, 0.9, summary_text, transform=ax6.transAxes, 
             fontsize=10, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*70)
    print("VISUALIZATION COMPLETE")
    print("="*70)
    
else:
    print("\n⚠️  No model results available yet. Run the model training cells first.")
    print("\nTo generate complete comparison:")
    print("  1. Run Cell 22: Confusion Matrix (Isolation Forest)")
    print("  2. Run Cells 24-26: Baseline supervised models")
    print("  3. Run Cells 27-30: Optimized models with SMOTE + GridSearch")

In [0]:
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_curve, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("="*70)
print("PR-AUC vs ROC-AUC COMPREHENSIVE COMPARISON")
print("="*70)
print("\nWhy PR-AUC matters for fraud detection:")
print("  - ROC-AUC can be overly optimistic with imbalanced datasets")
print("  - PR-AUC focuses on the minority class (fraud) performance")
print("  - Better metric when false positives are costly")
print("="*70)

# Collect all metrics
all_metrics_data = []

# 1. ISOLATION FOREST (Unsupervised)
print("\n[1] Isolation Forest (Unsupervised):")
if 'eval_df' in globals():
    iso_fraud_prob = 1 / (1 + np.exp(eval_df['anomaly_score']))
    iso_roc_auc = roc_auc_score(eval_df['actual_fraud'], iso_fraud_prob)
    iso_pr_auc = average_precision_score(eval_df['actual_fraud'], iso_fraud_prob)
    print(f"    ROC-AUC: {iso_roc_auc:.4f}")
    print(f"    PR-AUC:  {iso_pr_auc:.4f}")
    all_metrics_data.append({
        'Model': 'Isolation Forest',
        'Category': 'Unsupervised',
        'ROC-AUC': iso_roc_auc,
        'PR-AUC': iso_pr_auc,
        'Type': 'Baseline',
        'y_true': eval_df['actual_fraud'],
        'y_proba': iso_fraud_prob
    })
else:
    print("    Note: Run confusion matrix cell first")

# 2. BASELINE SUPERVISED MODELS
print("\n[2] Baseline Supervised Models:")
if 'comparison_df' in globals() and 'proba_predictions' in globals() and 'y_test' in globals():
    for model_name, y_proba in proba_predictions.items():
        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)
        print(f"    {model_name}:")
        print(f"      ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}")
        all_metrics_data.append({
            'Model': model_name,
            'Category': 'Supervised (Baseline)',
            'ROC-AUC': roc_auc,
            'PR-AUC': pr_auc,
            'Type': 'Baseline',
            'y_true': y_test,
            'y_proba': y_proba
        })
else:
    print("    Note: Run baseline model training first")

# 3. OPTIMIZED SUPERVISED MODELS
print("\n[3] Optimized Supervised Models (SMOTE + GridSearch):")
if 'best_models' in globals() and 'y_test' in globals():
    for model_name, model_info in best_models.items():
        y_proba = model_info['probabilities']
        roc_auc = roc_auc_score(y_test, y_proba)
        pr_auc = average_precision_score(y_test, y_proba)
        short_name = model_name.replace(' (GridSearch)', '')
        print(f"    {short_name}:")
        print(f"      ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}")
        all_metrics_data.append({
            'Model': short_name,
            'Category': 'Supervised (Optimized)',
            'ROC-AUC': roc_auc,
            'PR-AUC': pr_auc,
            'Type': 'Optimized',
            'y_true': y_test,
            'y_proba': y_proba
        })
else:
    print("    Note: Run optimized model training first")

if len(all_metrics_data) > 0:
    # Create comparison dataframe
    metrics_df = pd.DataFrame([{k: v for k, v in d.items() if k not in ['y_true', 'y_proba']} 
                               for d in all_metrics_data])
    
    print("\n" + "="*70)
    print("METRICS COMPARISON TABLE")
    print("="*70)
    display(metrics_df.sort_values('PR-AUC', ascending=False))
    
    # Find best models
    best_roc_idx = metrics_df['ROC-AUC'].idxmax()
    best_pr_idx = metrics_df['PR-AUC'].idxmax()
    
    best_roc_model = metrics_df.loc[best_roc_idx, 'Model']
    best_roc_score = metrics_df.loc[best_roc_idx, 'ROC-AUC']
    
    best_pr_model = metrics_df.loc[best_pr_idx, 'Model']
    best_pr_score = metrics_df.loc[best_pr_idx, 'PR-AUC']
    
    print("\n" + "="*70)
    print(f"BEST MODEL BY ROC-AUC: {best_roc_model} (Score: {best_roc_score:.4f})")
    print(f"BEST MODEL BY PR-AUC:  {best_pr_model} (Score: {best_pr_score:.4f})")
    if best_roc_model != best_pr_model:
        print("\n⚠️  Note: Different models excel at different metrics!")
        print("    For fraud detection, prioritize PR-AUC (focuses on fraud class)")
    print("="*70)
    
    # COMPREHENSIVE VISUALIZATIONS
    fig = plt.figure(figsize=(20, 16))
    
    # 1. ROC-AUC vs PR-AUC Scatter Plot
    ax1 = plt.subplot(3, 3, 1)
    colors_map = {'Unsupervised': '#e74c3c', 'Supervised (Baseline)': '#3498db', 
                  'Supervised (Optimized)': '#2ecc71'}
    for category in metrics_df['Category'].unique():
        cat_data = metrics_df[metrics_df['Category'] == category]
        ax1.scatter(cat_data['ROC-AUC'], cat_data['PR-AUC'], 
                   label=category, s=150, alpha=0.7, color=colors_map[category])
    
    # Add model labels
    for idx, row in metrics_df.iterrows():
        ax1.annotate(row['Model'].replace('Logistic Regression', 'LR').replace('Random Forest', 'RF'),
                    (row['ROC-AUC'], row['PR-AUC']), 
                    xytext=(5, 5), textcoords='offset points', fontsize=8)
    
    ax1.set_xlabel('ROC-AUC')
    ax1.set_ylabel('PR-AUC')
    ax1.set_title('ROC-AUC vs PR-AUC\n(Top-right = Best)')
    ax1.legend()
    ax1.grid(alpha=0.3)
    ax1.set_xlim([0.5, 1.0])
    
    # 2. Side-by-side Bar Comparison
    ax2 = plt.subplot(3, 3, 2)
    sorted_df = metrics_df.sort_values('PR-AUC', ascending=True)
    x_pos = np.arange(len(sorted_df))
    width = 0.35
    
    ax2.barh(x_pos - width/2, sorted_df['ROC-AUC'], width, label='ROC-AUC', color='skyblue', alpha=0.8)
    ax2.barh(x_pos + width/2, sorted_df['PR-AUC'], width, label='PR-AUC', color='coral', alpha=0.8)
    
    ax2.set_yticks(x_pos)
    ax2.set_yticklabels(sorted_df['Model'])
    ax2.set_xlabel('Score')
    ax2.set_title('ROC-AUC vs PR-AUC Comparison')
    ax2.legend()
    ax2.set_xlim([0, 1.0])
    ax2.grid(axis='x', alpha=0.3)
    
    # 3. Metric Difference (ROC-AUC - PR-AUC)
    ax3 = plt.subplot(3, 3, 3)
    metrics_df['Difference'] = metrics_df['ROC-AUC'] - metrics_df['PR-AUC']
    sorted_diff = metrics_df.sort_values('Difference', ascending=True)
    colors_diff = ['red' if x > 0.1 else 'orange' if x > 0.05 else 'green' 
                   for x in sorted_diff['Difference']]
    
    ax3.barh(range(len(sorted_diff)), sorted_diff['Difference'], color=colors_diff, alpha=0.7)
    ax3.set_yticks(range(len(sorted_diff)))
    ax3.set_yticklabels(sorted_diff['Model'])
    ax3.set_xlabel('ROC-AUC - PR-AUC')
    ax3.set_title('Metric Gap\n(Larger gap = more imbalanced performance)')
    ax3.axvline(x=0, color='black', linestyle='--', linewidth=1)
    for i, v in enumerate(sorted_diff['Difference']):
        ax3.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)
    ax3.grid(axis='x', alpha=0.3)
    
    # 4. Combined ROC Curves
    ax4 = plt.subplot(3, 3, 4)
    for data in all_metrics_data:
        if 'y_true' in data and 'y_proba' in data:
            fpr, tpr, _ = roc_curve(data['y_true'], data['y_proba'])
            ax4.plot(fpr, tpr, label=f"{data['Model']} ({data['ROC-AUC']:.3f})", linewidth=2)
    
    ax4.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (0.500)')
    ax4.set_xlabel('False Positive Rate')
    ax4.set_ylabel('True Positive Rate')
    ax4.set_title('ROC Curves - All Models')
    ax4.legend(loc='lower right', fontsize=8)
    ax4.grid(alpha=0.3)
    ax4.set_xlim([0, 1])
    ax4.set_ylim([0, 1.05])
    
    # 5. Combined Precision-Recall Curves
    ax5 = plt.subplot(3, 3, 5)
    for data in all_metrics_data:
        if 'y_true' in data and 'y_proba' in data:
            precision, recall, _ = precision_recall_curve(data['y_true'], data['y_proba'])
            ax5.plot(recall, precision, label=f"{data['Model']} ({data['PR-AUC']:.3f})", linewidth=2)
    
    # Baseline (proportion of positives)
    if len(all_metrics_data) > 0:
        baseline_rate = all_metrics_data[0]['y_true'].mean()
        ax5.axhline(y=baseline_rate, color='k', linestyle='--', linewidth=1, 
                   label=f'Baseline ({baseline_rate:.3f})')
    
    ax5.set_xlabel('Recall')
    ax5.set_ylabel('Precision')
    ax5.set_title('Precision-Recall Curves - All Models')
    ax5.legend(loc='best', fontsize=8)
    ax5.grid(alpha=0.3)
    ax5.set_xlim([0, 1])
    ax5.set_ylim([0, 1.05])
    
    # 6. PR-AUC by Category
    ax6 = plt.subplot(3, 3, 6)
    category_pr = metrics_df.groupby('Category')['PR-AUC'].agg(['mean', 'max', 'min'])
    x_pos = np.arange(len(category_pr))
    ax6.bar(x_pos, category_pr['mean'], alpha=0.6, label='Mean', color='steelblue')
    ax6.scatter(x_pos, category_pr['max'], color='green', s=100, marker='^', label='Max', zorder=3)
    ax6.scatter(x_pos, category_pr['min'], color='red', s=100, marker='v', label='Min', zorder=3)
    ax6.set_xticks(x_pos)
    ax6.set_xticklabels(category_pr.index, rotation=45, ha='right')
    ax6.set_ylabel('PR-AUC Score')
    ax6.set_title('PR-AUC by Model Category')
    ax6.legend()
    ax6.grid(axis='y', alpha=0.3)
    
    # 7. Improvement Analysis (Baseline vs Optimized)
    ax7 = plt.subplot(3, 3, 7)
    if 'comparison_df' in globals() and 'optimized_df' in globals():
        improvement_data = []
        for model in ['Logistic Regression', 'Random Forest', 'XGBoost']:
            baseline_data = metrics_df[(metrics_df['Model'] == model) & 
                                       (metrics_df['Category'] == 'Supervised (Baseline)')]
            opt_data = metrics_df[(metrics_df['Model'] == model) & 
                                  (metrics_df['Category'] == 'Supervised (Optimized)')]
            
            if not baseline_data.empty and not opt_data.empty:
                improvement_data.append({
                    'Model': model,
                    'Baseline PR-AUC': baseline_data['PR-AUC'].values[0],
                    'Optimized PR-AUC': opt_data['PR-AUC'].values[0],
                    'Improvement': (opt_data['PR-AUC'].values[0] - baseline_data['PR-AUC'].values[0]) * 100
                })
        
        if improvement_data:
            imp_df = pd.DataFrame(improvement_data)
            x_pos = np.arange(len(imp_df))
            width = 0.35
            
            ax7.bar(x_pos - width/2, imp_df['Baseline PR-AUC'], width, 
                   label='Baseline', color='lightcoral', alpha=0.8)
            ax7.bar(x_pos + width/2, imp_df['Optimized PR-AUC'], width, 
                   label='Optimized', color='lightgreen', alpha=0.8)
            
            ax7.set_xticks(x_pos)
            ax7.set_xticklabels([m.replace('Logistic Regression', 'LR').replace('Random Forest', 'RF') 
                                for m in imp_df['Model']], rotation=45, ha='right')
            ax7.set_ylabel('PR-AUC')
            ax7.set_title('PR-AUC: Baseline vs Optimized')
            ax7.legend()
            ax7.grid(axis='y', alpha=0.3)
            
            # Add improvement labels
            for i, row in imp_df.iterrows():
                if row['Improvement'] > 0:
                    ax7.text(i, max(row['Baseline PR-AUC'], row['Optimized PR-AUC']) + 0.02,
                            f"+{row['Improvement']:.1f}%", ha='center', fontsize=9, 
                            color='green', weight='bold')
    
    # 8. Ranking Comparison
    ax8 = plt.subplot(3, 3, 8)
    metrics_df['ROC_Rank'] = metrics_df['ROC-AUC'].rank(ascending=False)
    metrics_df['PR_Rank'] = metrics_df['PR-AUC'].rank(ascending=False)
    metrics_df['Rank_Diff'] = metrics_df['ROC_Rank'] - metrics_df['PR_Rank']
    
    sorted_rank = metrics_df.sort_values('PR_Rank')
    colors_rank = ['red' if x < -1 else 'orange' if x < 0 else 'lightgreen' if x > 0 else 'gray'
                   for x in sorted_rank['Rank_Diff']]
    
    ax8.barh(range(len(sorted_rank)), sorted_rank['Rank_Diff'], color=colors_rank, alpha=0.7)
    ax8.set_yticks(range(len(sorted_rank)))
    ax8.set_yticklabels(sorted_rank['Model'])
    ax8.set_xlabel('Rank Change (ROC → PR)')
    ax8.set_title('Ranking Shifts\n(Negative = Better PR-AUC Ranking)')
    ax8.axvline(x=0, color='black', linestyle='--', linewidth=1)
    ax8.grid(axis='x', alpha=0.3)
    
    # 9. Statistical Summary
    ax9 = plt.subplot(3, 3, 9)
    ax9.axis('off')
    
    summary_text = f"""
    STATISTICAL SUMMARY
    {'='*45}
    
    Total Models: {len(metrics_df)}
    
    Best by ROC-AUC:
      • {best_roc_model}
      • Score: {best_roc_score:.4f}
    
    Best by PR-AUC:
      • {best_pr_model}
      • Score: {best_pr_score:.4f}
    
    Average Metrics:
      • Mean ROC-AUC: {metrics_df['ROC-AUC'].mean():.4f}
      • Mean PR-AUC: {metrics_df['PR-AUC'].mean():.4f}
      • Mean Gap: {metrics_df['Difference'].mean():.4f}
    
    Category Performance (PR-AUC):
    """
    
    for category in metrics_df['Category'].unique():
        cat_mean = metrics_df[metrics_df['Category'] == category]['PR-AUC'].mean()
        summary_text += f"\n      • {category}: {cat_mean:.4f}"
    
    summary_text += "\n\n    RECOMMENDATION:"
    summary_text += f"\n      Deploy: {best_pr_model}"
    summary_text += f"\n      (Best PR-AUC for fraud detection)"
    
    ax9.text(0.05, 0.95, summary_text, transform=ax9.transAxes, 
             fontsize=9, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*70)
    print("KEY INSIGHTS")
    print("="*70)
    print(f"\n1. Metric Gap Analysis:")
    print(f"   Average ROC-AUC - PR-AUC gap: {metrics_df['Difference'].mean():.4f}")
    print(f"   This reflects class imbalance severity")
    
    print(f"\n2. Best Model for Production:")
    print(f"   {best_pr_model} with PR-AUC = {best_pr_score:.4f}")
    print(f"   (Prioritize PR-AUC for fraud detection)")
    
    print(f"\n3. Optimization Impact:")
    if 'improvement_data' in locals() and improvement_data:
        avg_improvement = np.mean([d['Improvement'] for d in improvement_data])
        print(f"   SMOTE + GridSearch improved PR-AUC by {avg_improvement:.1f}% on average")
    
    print("\n" + "="*70)
    
else:
    print("\n⚠️  No model results available. Run model training cells first.")

In [0]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import time

# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Set MLflow experiment
mlflow.set_experiment("/Users/ibritics@gmail.com/payment-analysis/fraud-detection-supervised")

# Store models and predictions
models = {}
predictions = {}
proba_predictions = {}

print("="*60)
print("TRAINING SUPERVISED LEARNING MODELS")
print("="*60)

# 1. LOGISTIC REGRESSION
print("\n[1/3] Training Logistic Regression...")
start_time = time.time()

with mlflow.start_run(run_name="logistic_regression") as run:
    lr_model = LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced',  # Handle class imbalance
        n_jobs=-1
    )
    lr_model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred_lr = lr_model.predict(X_test_scaled)
    y_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]
    
    # Metrics
    roc_auc_lr = roc_auc_score(y_test, y_proba_lr)
    cm_lr = confusion_matrix(y_test, y_pred_lr)
    tn, fp, fn, tp = cm_lr.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Log to MLflow
    mlflow.log_params({
        "model": "LogisticRegression",
        "max_iter": 1000,
        "class_weight": "balanced"
    })
    mlflow.log_metrics({
        "roc_auc": roc_auc_lr,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn
    })
    mlflow.sklearn.log_model(lr_model, "model")
    
    models['Logistic Regression'] = lr_model
    predictions['Logistic Regression'] = y_pred_lr
    proba_predictions['Logistic Regression'] = y_proba_lr
    
    elapsed = time.time() - start_time
    print(f"   Completed in {elapsed:.2f}s | ROC-AUC: {roc_auc_lr:.4f} | F1: {f1:.4f}")

# 2. RANDOM FOREST
print("\n[2/3] Training Random Forest...")
start_time = time.time()

with mlflow.start_run(run_name="random_forest") as run:
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    
    # Predictions
    y_pred_rf = rf_model.predict(X_test)
    y_proba_rf = rf_model.predict_proba(X_test)[:, 1]
    
    # Metrics
    roc_auc_rf = roc_auc_score(y_test, y_proba_rf)
    cm_rf = confusion_matrix(y_test, y_pred_rf)
    tn, fp, fn, tp = cm_rf.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Log to MLflow
    mlflow.log_params({
        "model": "RandomForest",
        "n_estimators": 100,
        "max_depth": 10,
        "class_weight": "balanced"
    })
    mlflow.log_metrics({
        "roc_auc": roc_auc_rf,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn
    })
    mlflow.sklearn.log_model(rf_model, "model")
    
    models['Random Forest'] = rf_model
    predictions['Random Forest'] = y_pred_rf
    proba_predictions['Random Forest'] = y_proba_rf
    
    elapsed = time.time() - start_time
    print(f"   Completed in {elapsed:.2f}s | ROC-AUC: {roc_auc_rf:.4f} | F1: {f1:.4f}")

# 3. XGBOOST
print("\n[3/3] Training XGBoost...")
start_time = time.time()

with mlflow.start_run(run_name="xgboost") as run:
    # Calculate scale_pos_weight for class imbalance
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
    
    xgb_model = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss'
    )
    xgb_model.fit(X_train, y_train)
    
    # Predictions
    y_pred_xgb = xgb_model.predict(X_test)
    y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
    
    # Metrics
    roc_auc_xgb = roc_auc_score(y_test, y_proba_xgb)
    cm_xgb = confusion_matrix(y_test, y_pred_xgb)
    tn, fp, fn, tp = cm_xgb.ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Log to MLflow
    mlflow.log_params({
        "model": "XGBoost",
        "n_estimators": 100,
        "max_depth": 6,
        "learning_rate": 0.1,
        "scale_pos_weight": scale_pos_weight
    })
    mlflow.log_metrics({
        "roc_auc": roc_auc_xgb,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn
    })
    mlflow.xgboost.log_model(xgb_model, "model")
    
    models['XGBoost'] = xgb_model
    predictions['XGBoost'] = y_pred_xgb
    proba_predictions['XGBoost'] = y_proba_xgb
    
    elapsed = time.time() - start_time
    print(f"   Completed in {elapsed:.2f}s | ROC-AUC: {roc_auc_xgb:.4f} | F1: {f1:.4f}")

print("\n" + "="*60)
print("ALL MODELS TRAINED SUCCESSFULLY")
print("="*60)

In [0]:
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score, 
                             precision_recall_curve, average_precision_score,
                             precision_score, recall_score, f1_score)
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

print("="*70)
print("SUPERVISED VS UNSUPERVISED MODEL COMPARISON")
print("="*70)

# ===== SUPERVISED MODELS =====
supervised_results = []

for model_name in ['Logistic Regression', 'Random Forest', 'XGBoost']:
    y_pred = predictions[model_name]
    y_proba = proba_predictions[model_name]
    
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    
    supervised_results.append({
        'Model': model_name,
        'Type': 'Supervised',
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'PR-AUC': pr_auc,
        'y_true': y_test,
        'y_pred': y_pred,
        'y_proba': y_proba
    })
    
    print(f"\n{model_name} (Supervised):")
    print(f"  Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f} | PR-AUC: {pr_auc:.4f}")

# ===== UNSUPERVISED MODEL (ISOLATION FOREST) =====
print("\n" + "="*70)
print("Loading Isolation Forest results...")
print("="*70)

# Note: Isolation Forest predictions need to be loaded from earlier cells
# If eval_df exists from the Isolation Forest section, use it
if 'eval_df' in globals():
    iso_y_true = eval_df['actual_fraud']
    iso_y_pred = eval_df['is_fraud']
    iso_y_proba = -eval_df['anomaly_score']  # Negative because lower score = more anomalous
    
    iso_precision = precision_score(iso_y_true, iso_y_pred)
    iso_recall = recall_score(iso_y_true, iso_y_pred)
    iso_f1 = f1_score(iso_y_true, iso_y_pred)
    iso_roc_auc = roc_auc_score(iso_y_true, iso_y_proba)
    iso_pr_auc = average_precision_score(iso_y_true, iso_y_proba)
    
    supervised_results.append({
        'Model': 'Isolation Forest',
        'Type': 'Unsupervised',
        'Precision': iso_precision,
        'Recall': iso_recall,
        'F1-Score': iso_f1,
        'ROC-AUC': iso_roc_auc,
        'PR-AUC': iso_pr_auc,
        'y_true': iso_y_true,
        'y_pred': iso_y_pred,
        'y_proba': iso_y_proba
    })
    
    print(f"\nIsolation Forest (Unsupervised):")
    print(f"  Precision: {iso_precision:.4f} | Recall: {iso_recall:.4f} | F1: {iso_f1:.4f}")
    print(f"  ROC-AUC: {iso_roc_auc:.4f} | PR-AUC: {iso_pr_auc:.4f}")
else:
    print("\n⚠️  Isolation Forest results not found. Run the Isolation Forest cells first.")

# Create comparison dataframe
all_models_df = pd.DataFrame(supervised_results)

print("\n" + "="*70)
print("SUMMARY TABLE")
print("="*70)
display(all_models_df[['Model', 'Type', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'PR-AUC']])

# ===== VISUALIZATIONS =====
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Define colors
colors = {'Supervised': ['#1f77b4', '#ff7f0e', '#2ca02c'], 
          'Unsupervised': ['#d62728']}

# 1. ROC CURVES
ax1 = axes[0, 0]
color_idx = {'Supervised': 0, 'Unsupervised': 0}

for idx, row in all_models_df.iterrows():
    model_type = row['Type']
    fpr, tpr, _ = roc_curve(row['y_true'], row['y_proba'])
    color = colors[model_type][color_idx[model_type]]
    color_idx[model_type] += 1
    
    linestyle = '-' if model_type == 'Supervised' else '--'
    ax1.plot(fpr, tpr, color=color, lw=2.5, linestyle=linestyle,
             label=f"{row['Model']} ({model_type}) - AUC={row['ROC-AUC']:.3f}")

ax1.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate', fontsize=11)
ax1.set_ylabel('True Positive Rate', fontsize=11)
ax1.set_title('ROC Curves - All Models', fontsize=13, fontweight='bold')
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(alpha=0.3)

# 2. PRECISION-RECALL CURVES
ax2 = axes[0, 1]
color_idx = {'Supervised': 0, 'Unsupervised': 0}

for idx, row in all_models_df.iterrows():
    model_type = row['Type']
    precision_curve, recall_curve, _ = precision_recall_curve(row['y_true'], row['y_proba'])
    color = colors[model_type][color_idx[model_type]]
    color_idx[model_type] += 1
    
    linestyle = '-' if model_type == 'Supervised' else '--'
    ax2.plot(recall_curve, precision_curve, color=color, lw=2.5, linestyle=linestyle,
             label=f"{row['Model']} ({model_type}) - AP={row['PR-AUC']:.3f}")

# Baseline (fraud rate)
fraud_rate = all_models_df.iloc[0]['y_true'].mean()
ax2.axhline(y=fraud_rate, color='k', linestyle='--', lw=1.5, label=f'Baseline (Fraud Rate={fraud_rate:.3f})')

ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Recall', fontsize=11)
ax2.set_ylabel('Precision', fontsize=11)
ax2.set_title('Precision-Recall Curves - All Models', fontsize=13, fontweight='bold')
ax2.legend(loc='best', fontsize=9)
ax2.grid(alpha=0.3)

# 3. PRECISION COMPARISON
ax3 = axes[1, 0]
models_list = all_models_df['Model'].tolist()
x_pos = np.arange(len(models_list))

# Color bars by type
bar_colors = []
for model_type in all_models_df['Type']:
    bar_colors.append('#1f77b4' if model_type == 'Supervised' else '#d62728')

bars = ax3.barh(x_pos, all_models_df['Precision'], color=bar_colors, alpha=0.8)
ax3.set_yticks(x_pos)
ax3.set_yticklabels(models_list)
ax3.set_xlabel('Precision', fontsize=11)
ax3.set_title('Precision Comparison\n(Higher = Fewer False Alarms)', fontsize=13, fontweight='bold')
ax3.set_xlim([0, 1])
ax3.grid(axis='x', alpha=0.3)

# Add value labels
for i, (v, model_type) in enumerate(zip(all_models_df['Precision'], all_models_df['Type'])):
    ax3.text(v + 0.02, i, f'{v:.3f}', va='center', fontsize=10)

ax3.invert_yaxis()

# 4. RECALL COMPARISON
ax4 = axes[1, 1]
bars = ax4.barh(x_pos, all_models_df['Recall'], color=bar_colors, alpha=0.8)
ax4.set_yticks(x_pos)
ax4.set_yticklabels(models_list)
ax4.set_xlabel('Recall', fontsize=11)
ax4.set_title('Recall Comparison\n(Higher = Fewer Missed Frauds)', fontsize=13, fontweight='bold')
ax4.set_xlim([0, 1])
ax4.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(all_models_df['Recall']):
    ax4.text(v + 0.02, i, f'{v:.3f}', va='center', fontsize=10)

ax4.invert_yaxis()

# Add legend for bar colors
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1f77b4', alpha=0.8, label='Supervised'),
                   Patch(facecolor='#d62728', alpha=0.8, label='Unsupervised')]
ax3.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)

best_roc = all_models_df.loc[all_models_df['ROC-AUC'].idxmax()]
best_pr = all_models_df.loc[all_models_df['PR-AUC'].idxmax()]
best_recall = all_models_df.loc[all_models_df['Recall'].idxmax()]

print(f"\n✓ Best ROC-AUC: {best_roc['Model']} ({best_roc['Type']}) = {best_roc['ROC-AUC']:.4f}")
print(f"✓ Best PR-AUC: {best_pr['Model']} ({best_pr['Type']}) = {best_pr['PR-AUC']:.4f}")
print(f"✓ Best Recall: {best_recall['Model']} ({best_recall['Type']}) = {best_recall['Recall']:.4f}")

print("\n💡 For imbalanced fraud detection:")
print("   - PR-AUC is more informative than ROC-AUC")
print("   - High recall is critical (catch more frauds)")
print("   - Balance precision vs recall based on business cost")

print("\n" + "="*70)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

print("="*70)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*70)

# Get feature importance from Random Forest and XGBoost
rf_importance = models['Random Forest'].feature_importances_
xgb_importance = models['XGBoost'].feature_importances_

# Create feature importance dataframe
feature_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Random Forest': rf_importance,
    'XGBoost': xgb_importance
})

# Calculate average importance
feature_importance_df['Average'] = (feature_importance_df['Random Forest'] + 
                                     feature_importance_df['XGBoost']) / 2

# Sort by average importance
feature_importance_df = feature_importance_df.sort_values('Average', ascending=False)

print("\nTop 15 Most Important Features:")
display(feature_importance_df.head(15))

# Visualizations
fig, axes = plt.subplots(1, 3, figsize=(20, 8))

# 1. Random Forest Feature Importance
ax1 = axes[0]
top_rf = feature_importance_df.nlargest(15, 'Random Forest')
ax1.barh(range(len(top_rf)), top_rf['Random Forest'], color='#2ca02c', alpha=0.8)
ax1.set_yticks(range(len(top_rf)))
ax1.set_yticklabels(top_rf['Feature'])
ax1.set_xlabel('Importance')
ax1.set_title('Random Forest\nTop 15 Features')
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3)

# 2. XGBoost Feature Importance
ax2 = axes[1]
top_xgb = feature_importance_df.nlargest(15, 'XGBoost')
ax2.barh(range(len(top_xgb)), top_xgb['XGBoost'], color='#ff7f0e', alpha=0.8)
ax2.set_yticks(range(len(top_xgb)))
ax2.set_yticklabels(top_xgb['Feature'])
ax2.set_xlabel('Importance')
ax2.set_title('XGBoost\nTop 15 Features')
ax2.invert_yaxis()
ax2.grid(axis='x', alpha=0.3)

# 3. Average Feature Importance (Combined)
ax3 = axes[2]
top_avg = feature_importance_df.nlargest(15, 'Average')
ax3.barh(range(len(top_avg)), top_avg['Average'], color='#1f77b4', alpha=0.8)
ax3.set_yticks(range(len(top_avg)))
ax3.set_yticklabels(top_avg['Feature'])
ax3.set_xlabel('Average Importance')
ax3.set_title('Combined Average\nTop 15 Features')
ax3.invert_yaxis()
ax3.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Group features by category
print("\n" + "="*70)
print("FEATURE IMPORTANCE BY CATEGORY")
print("="*70)

temporal_features = ['year', 'month', 'day', 'day_of_week', 'hour', 'is_weekend', 'is_night']
behavioral_features = ['user_avg_amount', 'user_stddev_amount', 'user_min_amount', 'user_max_amount',
                       'user_total_transactions', 'z_score_amount']
card_features = ['credit_limit', 'num_cards_issued', 'card_on_dark_web', 'card_brand_encoded', 'card_type_encoded']
transaction_features = ['amount', 'amount_to_limit_ratio', 'high_limit_usage', 'is_amount_outlier', 'amount_vs_user_avg_ratio']
aggregation_features = ['card_avg_amount', 'card_total_transactions', 'monthly_total_amount', 'monthly_transaction_count']

categories = {
    'Temporal': temporal_features,
    'Behavioral': behavioral_features,
    'Card-specific': card_features,
    'Transaction': transaction_features,
    'Aggregation': aggregation_features
}

for category_name, features in categories.items():
    category_importance = feature_importance_df[feature_importance_df['Feature'].isin(features)]['Average'].sum()
    print(f"{category_name:20s}: {category_importance:.4f} ({category_importance/feature_importance_df['Average'].sum()*100:.1f}%)")

print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)
top_3_features = feature_importance_df.head(3)['Feature'].tolist()
print(f"\nMost predictive features:")
for i, feat in enumerate(top_3_features, 1):
    importance = feature_importance_df[feature_importance_df['Feature'] == feat]['Average'].values[0]
    print(f"  {i}. {feat}: {importance:.4f}")

print("\nThese features have the strongest signal for detecting fraudulent transactions.")
print("Consider focusing on these when building real-time fraud detection systems.")

In [0]:
# Determine best model by F1-Score
best_model_name = comparison_df.loc[comparison_df['F1-Score'].idxmax(), 'Model']
best_model = models[best_model_name]
best_f1 = comparison_df['F1-Score'].max()
best_roc_auc = comparison_df.loc[comparison_df['Model'] == best_model_name, 'ROC-AUC'].values[0]

print("="*70)
print("SAVING BEST MODEL PREDICTIONS")
print("="*70)
print(f"\nBest Model: {best_model_name}")
print(f"F1-Score: {best_f1:.4f}")
print(f"ROC-AUC: {best_roc_auc:.4f}")

# Create final predictions dataframe with all test data
if best_model_name == 'Logistic Regression':
    final_predictions = best_model.predict(X_test_scaled)
    final_proba = best_model.predict_proba(X_test_scaled)[:, 1]
else:
    final_predictions = best_model.predict(X_test)
    final_proba = best_model.predict_proba(X_test)[:, 1]

# Create results dataframe
results_df = pd.DataFrame({
    'transaction_id': test_ids.values,
    'predicted_fraud': final_predictions,
    'fraud_probability': final_proba,
    'actual_fraud': y_test.values
})

# Add transaction details
results_df = results_df.merge(
    df_supervised[['id', 'client_id', 'card_id', 'amount', 'date']],
    left_on='transaction_id',
    right_on='id',
    how='left'
)

# Save to Spark table
results_spark = spark.createDataFrame(results_df)
results_spark.write.mode('overwrite').saveAsTable("workspace.default.supervised_fraud_predictions")

print(f"\n✓ Predictions saved to: workspace.default.supervised_fraud_predictions")
print(f"  Total predictions: {len(results_df):,}")
print(f"  Predicted frauds: {final_predictions.sum():,}")
print(f"  Actual frauds: {y_test.sum():,}")

# Show sample predictions (high fraud probability)
print("\n" + "="*70)
print("SAMPLE HIGH-RISK TRANSACTIONS (Top 10 by fraud probability)")
print("="*70)
high_risk = results_df.nlargest(10, 'fraud_probability')[[
    'transaction_id', 'client_id', 'amount', 'fraud_probability', 
    'predicted_fraud', 'actual_fraud'
]]
display(high_risk)

print("\n" + "="*70)
print("SUPERVISED LEARNING PIPELINE COMPLETE")
print("="*70)
print(f"\n✓ Trained 3 models: Logistic Regression, Random Forest, XGBoost")
print(f"✓ Best model: {best_model_name}")
print(f"✓ All models logged to MLflow")
print(f"✓ Predictions saved to supervised_fraud_predictions table")
print(f"\nNext Steps:")
print(f"  1. Review high-risk transactions flagged by {best_model_name}")
print(f"  2. Compare supervised vs unsupervised (Isolation Forest) results")
print(f"  3. Consider ensemble approach combining multiple models")
print(f"  4. Deploy best model for real-time fraud scoring")